# Ch 24 · CUDA Graphs: Capture, Replay and What Gets Baked

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/24-cuda-graphs.ipynb)

Companion notebook for the chapter page `chapters/24-cuda-graphs.html` ([page](../chapters/24-cuda-graphs.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

This notebook mirrors the chapter page: every example in page order (24.1, 24.2, 24.3, 24.6, 24.5, 24.8, 24.4, 24.7). It runs on a GPU runtime (big budgets) and on CPU (small budgets, `ON_GPU = False`). Numbers printed on CPU are CPU numbers, not GPU performance; on Colab you get your own GPU numbers.

## The running project: `my_env.py` (snapshot after Ch 24)
This chapter adds `needs_conditional_fallback(device)` and `capture_step(m, d, nsub=1, post=None)`. The cell writes the whole module so the notebook runs in an empty Colab runtime.

In [ ]:
%%writefile my_env.py
"""my_env.py - the course's running helper module, snapshot after Chapter 24.

Ch 6 added make_worlds. Ch 8 added reset_done, nonfinite_reset, snapshot and restore.
Ch 9 added describe_overflow (Example 9.7).
Ch 10 added randomize (Example 10.9): in-place, graph-friendly friction re-randomization.
Ch 14 added sensor_slices (Example 14.3): sensor name -> (adr, dim) in d.sensordata.
Ch 24 adds needs_conditional_fallback and capture_step (Example 24.4): record nsub steps (plus
optional user kernels) into one CUDA graph, with the graph-conditional fallback for CUDA < 12.4.
Later chapters extend this file (Ch 26 obs/reward kernels, Ch 30 the environments).
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

__all__ = ["make_worlds", "reset_done", "nonfinite_reset", "snapshot", "restore",
           "describe_overflow", "randomize", "sensor_slices", "needs_conditional_fallback",
           "capture_step"]


# --- Ch 6 ----------------------------------------------------------------------------------------
def make_worlds(xml, nworld, nconmax=None, njmax=None, batch_sizes=None, **sizes):
    """Compile `xml` and return (mjm, m, d) with `nworld` worlds (Ch 6 signature).

    nconmax / njmax are per world; batch_sizes (e.g. {"body_mass": nworld}) gives chosen Model
    fields their own rows (Ch 10); any other keyword (naconmax=, ...) goes to make_data.
    """
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm, batch_sizes=batch_sizes)
    d = mjw.make_data(mjm, nworld=nworld, nconmax=nconmax, njmax=njmax, **sizes)
    assert d.qpos.shape == (nworld, mjm.nq), d.qpos.shape
    return mjm, m, d


# --- Ch 8: resets --------------------------------------------------------------------------------
@wp.kernel
def _nonfinite_kernel(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                      done_out: wp.array[bool]):
    worldid = wp.tid()
    bad = bool(False)
    for i in range(qpos_in.shape[1]):
        if not wp.isfinite(qpos_in[worldid, i]):
            bad = True
    for i in range(qvel_in.shape[1]):
        if not wp.isfinite(qvel_in[worldid, i]):
            bad = True
    if bad:
        done_out[worldid] = True          # only ever sets flags; never clears another reason


def reset_done(m, d, done):
    """Reset the worlds whose done flag is True, then recompute derived quantities.

    `done` is a wp.array of bool with shape (nworld,). Everything stays on the device, so this
    can be captured in a CUDA graph together with mjw.step (Ch 24, Ch 26).
    """
    mjw.reset_data(m, d, reset=done)
    mjw.forward(m, d)                     # reset_data does not recompute xpos, sensors, ...


def nonfinite_reset(m, d, done):
    """OR `done` with "qpos or qvel has a NaN/Inf" (MJWarp has no mj_checkPos/mj_checkVel)."""
    wp.launch(_nonfinite_kernel, dim=d.nworld, inputs=[d.qpos, d.qvel], outputs=[done])


# --- Ch 8: state snapshots -----------------------------------------------------------------------
def snapshot(m, d, mjm, sig, buf=None):
    """Copy the state components in `sig` of every world into a (nworld, size) float array.

    Use State.PHYSICS for MPC (it includes HISTORY and PLASTIC; note it has no TIME bit),
    add State.WARMSTART for bitwise replays with contacts, State.INTEGRATION for everything.
    """
    size = mujoco.mj_stateSize(mjm, int(sig))
    if buf is None:
        buf = wp.zeros((d.nworld, size), dtype=float)
    # get_state does NOT check the size: an undersized buffer is written out of bounds.
    assert buf.shape == (d.nworld, size), f"state buffer {buf.shape} != {(d.nworld, size)}"
    mjw.get_state(m, d, buf, int(sig))
    return buf


def restore(m, d, mjm, buf, sig, active=None):
    """Write a snapshot back (only where the per-world bool mask `active` is True, if given)."""
    assert buf.shape == (d.nworld, mujoco.mj_stateSize(mjm, int(sig)))
    mjw.set_state(m, d, buf, int(sig), active)


# --- Ch 9: overflow ------------------------------------------------------------------------------
def describe_overflow(d, max_worlds=8):
    """Decode d.overflow: "none", or one line per flagged world plus a world count per bit.

    Loops over mjw.OverflowType (Python >= 3.11 iterates the single-bit members only), so bits
    added by a later MJWarp release are decoded without editing this function.
    """
    ov = d.overflow.numpy()
    if not ov.any():
        return "none"
    lines = []
    flagged = np.nonzero(ov)[0]
    for w in flagged[:max_worlds]:
        names = [bit.name for bit in mjw.OverflowType if ov[w] & bit]
        lines.append(f"world {w}: " + " | ".join(names))
    if len(flagged) > max_worlds:
        lines.append(f"... and {len(flagged) - max_worlds} more flagged worlds")
    counts = [f"{bit.name} x{int(np.count_nonzero(ov & bit))}" for bit in mjw.OverflowType
              if (ov & bit).any()]
    lines.append("worlds per bit: " + ", ".join(counts))
    return "\n".join(lines)


# --- Ch 10: domain randomization ----------------------------------------------------------------
@wp.kernel
def _randomize_friction_kernel(
    seed: wp.array(dtype=int),          # device array: a graph replays with the CURRENT value
    episode: wp.array(dtype=int),       # per-world episode counter, bumped on every re-draw
    reset: wp.array(dtype=bool),        # which worlds to re-draw (all if every entry is True)
    lo: float,
    hi: float,
    geom_friction: wp.array2d(dtype=wp.vec3),   # m.geom_friction, shape (nworld, ngeom)
):
    worldid, geomid = wp.tid()
    if not reset[worldid]:
        return
    # one random stream per (seed, world, episode): reproducible and different every episode
    rng = wp.rand_init(seed[0], worldid * 7919 + episode[worldid] * 104729)
    mu = wp.randf(rng, lo, hi)
    f = geom_friction[worldid, geomid]
    geom_friction[worldid, geomid] = wp.vec3(mu, f[1], f[2])


@wp.kernel
def _bump_episode_kernel(reset: wp.array(dtype=bool), episode: wp.array(dtype=int)):
    worldid = wp.tid()
    if reset[worldid]:
        episode[worldid] = episode[worldid] + 1


def randomize(m, seed, episode, reset, lo=0.1, hi=1.0):
    """Draw a new sliding friction in [lo, hi) for every geom of the worlds marked in reset.

    Writes m.geom_friction in place (it must already have one row per world, e.g. from
    put_model(batch_sizes={"geom_friction": nworld})), so a captured CUDA graph that contains
    this call keeps working: only array CONTENTS change, never array objects.
    seed: wp.array(dtype=int, shape=(1,)); episode: wp.array(dtype=int, shape=(nworld,));
    reset: wp.array(dtype=bool, shape=(nworld,)).
    """
    nworld = reset.shape[0]
    assert m.geom_friction.shape[0] == nworld, "batch geom_friction to nworld first"
    wp.launch(_randomize_friction_kernel, dim=(nworld, m.ngeom),
              inputs=[seed, episode, reset, lo, hi], outputs=[m.geom_friction])
    wp.launch(_bump_episode_kernel, dim=nworld, inputs=[reset], outputs=[episode])


# --- Ch 14: sensors ------------------------------------------------------------------------------
def sensor_slices(mjm):
    """Return {sensor name: (adr, dim)} so d.sensordata[:, adr:adr + dim] is that sensor.

    Unnamed sensors get the key "sensor<i>". The addresses come from the compiled MjModel
    (mjm.sensor_adr, mjm.sensor_dim) and are the same in every world.
    """
    out = {}
    for i in range(mjm.nsensor):
        name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_SENSOR, i) or f"sensor{i}"
        out[name] = (int(mjm.sensor_adr[i]), int(mjm.sensor_dim[i]))
    return out


# --- Ch 24: CUDA graphs ---------------------------------------------------------------------------
def needs_conditional_fallback(device=None):
    """True when `device` is a CUDA device whose toolkit or driver is older than CUDA 12.4.

    There, CUDA graph conditional nodes (wp.capture_while / wp.capture_if) do not exist, and
    put_model only *warns* (warp_util.check_toolkit_driver) while m.opt.graph_conditional stays True.
    On the CPU device the answer is False: Warp 1.19 records CPU graphs with APIC, which replays
    capture_while / capture_if callbacks without CUDA.
    """
    device = wp.get_device(device)
    return device.is_cuda and not wp.is_conditional_graph_supported()


def capture_step(m, d, nsub=1, post=None, verbose=True):
    """Record `nsub` calls of mjw.step(m, d), then `post(m, d)` if given, into one graph.

    Returns the graph; replay it with wp.capture_launch(graph). Capturing records the launches,
    it does not run them, so d is not advanced. The graph holds the *addresses* of every array it
    touches and the Python decisions taken while recording (integrator, solver, iteration
    counts, flags): change array contents in place (assign, fill_, copy, kernels) and recapture
    after replacing an array or changing a Python-level option (the Ch 24 knob board).
    """
    assert nsub >= 1, "nsub must be >= 1"
    device = d.qpos.device
    if needs_conditional_fallback(device) and m.opt.graph_conditional:
        m.opt.graph_conditional = False     # fixed iteration counts instead of device-side loops
        if verbose:
            print("capture_step: CUDA < 12.4, set m.opt.graph_conditional = False")
    with wp.ScopedCapture(device=device) as capture:
        for _ in range(nsub):
            mjw.step(m, d)
        if post is not None:
            post(m, d)
    return capture.graph

## 1. Capture once, replay many times
Warm up (compile), record one `mjw.step` inside `wp.ScopedCapture()`, then replay with `wp.capture_launch(graph)`. Watch `d.time`: capturing records the launches and does not run them.

In [ ]:
# Example 24.1 - ex1_capture.py
"""Example 24.1 - warm up, capture one mjw.step into a graph, replay it 5 times.

Capturing RECORDS the kernel launches of one step; it does not run them, so d.time is the same
before and after the `with wp.ScopedCapture()` block. Each wp.capture_launch(graph) then runs
the whole recorded step with one call. We also count the launches recorded during capture by
wrapping wp.launch / wp.launch_tiled (the same trick as Example 7.5).
Works on CPU (Warp 1.19 records CPU graphs) and on a CUDA GPU without changes.
Run: python ex1_capture.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
d.qpos.assign(np.tile(mjm.key_qpos[0], (NWORLD, 1)))      # start horizontal in every world

# Rule G: CUDA < 12.4 has no conditional graph nodes; fall back to fixed iteration counts.
if wp.get_device().is_cuda and not wp.is_conditional_graph_supported():
    m.opt.graph_conditional = False
    print("note: CUDA < 12.4, using m.opt.graph_conditional = False")


def t0():
    return f"{d.time.numpy()[0]:.2f}"


print("device:", wp.get_device(), "| nworld:", NWORLD)
print("after make_data:        d.time =", t0())

mjw.step(m, d)                                  # 1. warm-up: compiles (or loads) every kernel
print("after 1 warm-up step:   d.time =", t0())

# count launches while capturing (wrap the two launch functions MJWarp calls)
count = {"launch": 0, "launch_tiled": 0}
orig = {name: getattr(wp, name) for name in count}


def counting(name):
    def wrapper(*args, **kwargs):
        count[name] += 1
        return orig[name](*args, **kwargs)
    return wrapper


for name in count:
    setattr(wp, name, counting(name))
try:
    with wp.ScopedCapture() as capture:         # 2. record one step
        mjw.step(m, d)
finally:
    for name in count:
        setattr(wp, name, orig[name])
graph = capture.graph
print("after capture:          d.time =", t0(), "(recorded, not executed)")
print(f"launches recorded: {count['launch']} wp.launch + {count['launch_tiled']} "
      f"wp.launch_tiled")

for i in range(5):                              # 3. replay: one call runs the whole step
    wp.capture_launch(graph)
    print(f"after replay {i + 1}:         d.time =", t0())
wp.synchronize()

print("hinge angles, world 0:", d.qpos.numpy()[0])
print("all worlds identical:", bool(np.all(d.qpos.numpy() == d.qpos.numpy()[0])))

## 2. Baked vs live
Array *contents* (`d.ctrl`, `m.opt.timestep`) are read at replay time. Python values (`m.opt.integrator`) were read while recording, so the graph ignores changes.

In [ ]:
# Example 24.2 - ex2_baked_vs_live.py
"""Example 24.2 - baked vs live: what a captured graph sees when you change things after capture.

We capture one step of PENDULUM, then change three things and replay 20 steps from the same
start state, comparing with eager mjw.step:
  1. d.ctrl            via .assign()   -> an array's CONTENTS: live, the graph uses it
  2. m.opt.timestep    via .assign()   -> also an array (one float per world): live
  3. m.opt.integrator  = RK4           -> a Python int read while recording: baked, ignored
Run: python ex2_baked_vs_live.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 20

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
if wp.get_device().is_cuda and not wp.is_conditional_graph_supported():
    m.opt.graph_conditional = False                    # Rule G fallback (CUDA < 12.4)
    print("note: CUDA < 12.4, using m.opt.graph_conditional = False")

q0 = np.tile(mjm.key_qpos[0], (NWORLD, 1))             # start horizontal, at rest, t = 0


def restart():
    """Put every world back at the start state (in place: the graph keeps working)."""
    d.qpos.assign(q0)
    d.qvel.zero_()
    d.time.zero_()


def rollout(use_graph):
    restart()
    for _ in range(NSTEP):
        if use_graph:
            wp.capture_launch(graph)
        else:
            mjw.step(m, d)
    wp.synchronize()
    return d.qpos.numpy().copy(), float(d.time.numpy()[0])


mjw.step(m, d)                                         # warm-up (compile)
with wp.ScopedCapture() as capture:                    # recorded with ctrl 0, dt 0.01, EULER
    mjw.step(m, d)
graph = capture.graph


def report(label, eager, replay, eager_old=None):
    (qe, te), (qg, tg) = eager, replay
    print(f"--- {label}")
    print(f"  eager : qpos w0 {qe[0]}  t {te:.3f}")
    print(f"  graph : qpos w0 {qg[0]}  t {tg:.3f}")
    print(f"  max |graph - eager| = {np.abs(qg - qe).max():.2e}")
    if eager_old is not None:
        print(f"  max |graph - eager with OLD value| = {np.abs(qg - eager_old[0]).max():.2e}")


# 1. ctrl: assign new contents, a different torque per world
d.ctrl.assign(np.linspace(-2.0, 2.0, NWORLD).reshape(NWORLD, 1))
report("1. d.ctrl.assign(...)        (array contents)", rollout(False), rollout(True))

# 2. timestep: m.opt.timestep is a wp.array with one entry (shared by all worlds)
m.opt.timestep.assign([0.005])
report("2. m.opt.timestep.assign(...) (array contents)", rollout(False), rollout(True))

# 3. integrator: a Python int; mjw.step branches on it in Python while recording
euler = rollout(False)                                 # eager, still EULER
m.opt.integrator = int(mjw.IntegratorType.RK4)
report("3. m.opt.integrator = RK4    (Python int)", rollout(False), rollout(True), euler)

m.opt.integrator = int(mjw.IntegratorType.EULER)       # put it back before any recapture
print("verdict: ctrl live, timestep live, integrator baked (recapture after changing it)")

## 3. Replacing an array after capture
The graph holds addresses. `d.qpos = wp.array(...)` points `d` at new memory the graph never touches. Use `.assign()`, or recapture.

In [ ]:
# Example 24.3 - ex3_replace_after_capture.py
"""Example 24.3 - replacing an array after capture: the graph keeps writing the OLD array.

A graph stores the device addresses of the arrays it was recorded with. `d.qpos = wp.array(...)`
makes d point at a NEW array, but the graph still reads and writes the old one, so the new
array never changes and nothing warns you. Fix 1: change contents in place with .assign().
Fix 2: recapture after the replacement. (Ch 6 owns replace vs assign; this is the graph side.)
Run: python ex3_replace_after_capture.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NREPLAY = 50

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
if wp.get_device().is_cuda and not wp.is_conditional_graph_supported():
    m.opt.graph_conditional = False                    # Rule G fallback (CUDA < 12.4)
    print("note: CUDA < 12.4, using m.opt.graph_conditional = False")

mjw.step(m, d)                                         # warm-up
with wp.ScopedCapture() as capture:
    mjw.step(m, d)
graph = capture.graph

start = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
start[:, 2] = 2.0                                      # every ball starts at z = 2 m


def z(arr):
    return f"{arr.numpy()[0, 2]:.4f}"


# --- WRONG: replace the array object ------------------------------------------------------
old = d.qpos
d.qpos = wp.array(start, dtype=float)                  # new allocation, new address
d.qvel.zero_()
for _ in range(NREPLAY):
    wp.capture_launch(graph)
wp.synchronize()
print(f"replace: new d.qpos z = {z(d.qpos)}  (never written by the graph)")
print(f"         old array  z = {z(old)}  (the graph kept stepping this one)")
print("         same object?", d.qpos is old, "| same address?", d.qpos.ptr == old.ptr)

# --- RIGHT (fix 1): restore the old object and change its contents in place ---------------
d.qpos = old
d.qpos.assign(start)
d.qvel.zero_()
for _ in range(NREPLAY):
    wp.capture_launch(graph)
wp.synchronize()
print(f"assign:  d.qpos z = {z(d.qpos)}  (falls from 2.0 as expected)")

# --- RIGHT (fix 2): replace, then recapture so the graph records the new address ----------
d.qpos = wp.array(start, dtype=float)
d.qvel.zero_()
with wp.ScopedCapture() as capture:
    mjw.step(m, d)
graph = capture.graph
for _ in range(NREPLAY):
    wp.capture_launch(graph)
wp.synchronize()
print(f"replace + recapture: d.qpos z = {z(d.qpos)}  (matches the assign result)")

## 4. The knob board
This example counts the Warp modules that a change compiles, so it must run in a fresh Python process: the cell writes the file and runs it with `!python`.

In [ ]:
%%writefile ex6_knob_board.py
"""Example 24.6 - the knob board: which Option changes a captured graph sees.

For each knob we capture one step of box_stack(3) (CG solver, so the solver iterates), change
the knob, and run 10 steps three ways from the same saved state:
  graph   - replay the graph captured BEFORE the change
  new     - eager mjw.step with the new value
  old     - eager mjw.step with the old value (run before the change)
and count the Warp modules loaded by the first eager step after the change (new compiled code).
Verdict: live (graph == new), recapture (graph == old, no new code),
recompile + recapture (new code was compiled; the old graph cannot contain it).
The module count reads warp._src.context.user_modules (private Warp API, may change). Run it in a
fresh process (the notebook runs it with !python): a second run in the same Python session, or a
session that already used a value, finds those modules loaded (+mod 0).
Run: python ex6_knob_board.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from warp._src import context as wp_context      # private: only to count loaded modules

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 2
NSTEP = 10
SIG = int(mjw.State.INTEGRATION)

mjm = mujoco.MjModel.from_xml_string(box_stack(3))
mjm.opt.solver = mujoco.mjtSolver.mjSOL_CG       # CG iterates more than Newton: knobs show
mjm.opt.tolerance = 1e-10                        # put_model clamps this to 1e-6 (float32)


def set_array(arr, value):
    arr.fill_(value)                             # in place: same array object, new contents


KNOBS = [  # (field, what we do, function applying the change)
    ("timestep", "fill_(0.002)", lambda m: set_array(m.opt.timestep, 0.002)),
    ("gravity", "fill_((0, 0, -3))", lambda m: set_array(m.opt.gravity, wp.vec3(0.0, 0.0, -3.0))),
    ("tolerance", "fill_(0.1)", lambda m: set_array(m.opt.tolerance, 0.1)),
    ("iterations", "= 2", lambda m: setattr(m.opt, "iterations", 2)),
    ("ls_iterations", "= 1", lambda m: setattr(m.opt, "ls_iterations", 1)),
    ("integrator", "= RK4", lambda m: setattr(m.opt, "integrator", int(mjw.IntegratorType.RK4))),
    ("warn_overflow", "= ALL", lambda m: setattr(m.opt, "warn_overflow",
                                                  int(mjw.OverflowType.ALL))),
]


def loaded_modules():
    return sum(len(mod.execs) for mod in wp_context.user_modules.values())


def measure(change):
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0                      # quiet baseline (and a knob we test below)
    if wp.get_device().is_cuda and not wp.is_conditional_graph_supported():
        m.opt.graph_conditional = False          # Rule G fallback (CUDA < 12.4)
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
    for _ in range(20):                          # let the boxes land, then kick them
        mjw.step(m, d)
    rng = np.random.default_rng(0)
    qvel = d.qvel.numpy()
    qvel[:, 0::6] += rng.normal(0.0, 1.0, (NWORLD, 3))
    d.qvel.assign(qvel)
    state = wp.zeros((NWORLD, mujoco.mj_stateSize(mjm, SIG)), dtype=float)
    mjw.get_state(m, d, state, SIG)
    with wp.ScopedCapture() as capture:
        mjw.step(m, d)
    graph = capture.graph

    def rollout(fn):
        mjw.set_state(m, d, state, SIG)
        for _ in range(NSTEP):
            fn()
        wp.synchronize()
        return d.qpos.numpy().copy()

    old = rollout(lambda: mjw.step(m, d))
    change(m)
    replay = rollout(lambda: wp.capture_launch(graph))
    before = loaded_modules()
    new = rollout(lambda: mjw.step(m, d))
    return replay, new, old, loaded_modules() - before


rows = []
print(f"device: {wp.get_device()}, nworld {NWORLD}, {NSTEP} steps per comparison")
print(f"{'knob':<14} {'change':<17} {'|g-new|':>9} {'|g-old|':>9} {'|new-old|':>9} "
      f"{'+mod':>4}  verdict")
for field, what, change in KNOBS:
    g, new, old, nmod = measure(change)
    e_gn, e_go, e_no = (float(np.abs(a - b).max()) for a, b in ((g, new), (g, old), (new, old)))
    if e_no > 0 and e_gn <= 0.1 * e_no:
        verdict = "live"
    elif nmod > 0:
        verdict = "recompile + recapture"
    elif e_no > 0 and e_go <= 0.1 * e_no:
        verdict = "recapture"
    else:
        verdict = "no visible effect"
    rows.append({"field": field, "change": what, "graph_vs_new": e_gn, "graph_vs_old": e_go,
                 "new_vs_old": e_no, "new_modules": nmod, "verdict": verdict})
    print(f"{field:<14} {what:<17} {e_gn:>9.2e} {e_go:>9.2e} {e_no:>9.2e} {nmod:>4}  {verdict}")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"device": str(wp.get_device()), "nworld": NWORLD, "nstep": NSTEP,
                   "rows": [{k: (float(f"{v:.3g}") if isinstance(v, float) else v)
                             for k, v in r.items()} for r in rows]}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
!python ex6_knob_board.py

## 5. Conditional graph nodes: `graph_conditional` True vs False
With True the solver loop is one `wp.capture_while` node; with False the graph holds `m.opt.iterations` copies of the iteration. Compare launches recorded, mean `solver_niter`, time per step and the results.

In [ ]:
# Example 24.5 - ex5_graph_conditional.py
"""Example 24.5 - m.opt.graph_conditional True vs False on box_stack(5).

True (the default): solver.py records wp.capture_while(nsolving, while_body=_solver_iteration):
the device repeats the iteration until every world has converged. False: the graph holds
m.opt.iterations (100) unrolled copies of the iteration; converged worlds skip the work, but every
copy is still launched. We count the launches recorded, replay NSTEP steps, and compare the mean
d.solver_niter (in a second, untimed pass), the time per step / world-steps per second and the
final qpos.
CPU: Warp 1.19 records CPU graphs (APIC) and replays capture_while there too; timings printed on
CPU are CPU-dev-box numbers, not GPU performance. On CUDA < 12.4 only False can run (Rule G).
Run: python ex5_graph_conditional.py
"""
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 1000 if ON_GPU else 100
REPEAT = 3
COND_OK = (not ON_GPU) or wp.is_conditional_graph_supported()

print("device:", wp.get_device())
print("wp.is_conditional_graph_supported():", wp.is_conditional_graph_supported())
if not ON_GPU:
    print("CPU: conditional nodes replay through Warp's CPU graph recorder (no CUDA needed)")

mjm = mujoco.MjModel.from_xml_string(box_stack(5))
orig_launch, orig_tiled = wp.launch, wp.launch_tiled


def capture_counting(m, d):
    """Capture one step and count the wp.launch / wp.launch_tiled calls recorded."""
    n = [0]

    def launch(*a, **k):
        n[0] += 1
        return orig_launch(*a, **k)

    def tiled(*a, **k):
        n[0] += 1
        return orig_tiled(*a, **k)

    wp.launch, wp.launch_tiled = launch, tiled
    try:
        with wp.ScopedCapture() as capture:
            mjw.step(m, d)
    finally:
        wp.launch, wp.launch_tiled = orig_launch, orig_tiled
    return capture.graph, n[0]


results = {}
for gc in (True, False):
    if gc and not COND_OK:
        print("graph_conditional=True: skipped (CUDA < 12.4 has no conditional graph nodes)")
        continue
    m = mjw.put_model(mjm)
    m.opt.graph_conditional = gc
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
    mjw.step(m, d)                                   # warm-up
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)   # fresh state, same start
    graph, nlaunch = capture_counting(m, d)
    # timed passes: NSTEP replays between two synchronize calls, no host reads inside;
    # best of REPEAT runs, each from the same start (reset in place keeps the recorded addresses)
    sec = float("inf")
    for _ in range(REPEAT):
        mjw.reset_data(m, d)
        wp.synchronize()
        t = time.perf_counter()
        for _ in range(NSTEP):
            wp.capture_launch(graph)
        wp.synchronize()
        sec = min(sec, time.perf_counter() - t)
    ms, wsps = sec / NSTEP * 1e3, NWORLD * NSTEP / sec
    q_timed = d.qpos.numpy().copy()
    # untimed pass from the same start: reset in place (same addresses), replay, sample niter
    mjw.reset_data(m, d)
    niter = []
    for i in range(NSTEP):
        wp.capture_launch(graph)
        if i % 10 == 0:                              # a host read: fine here, this pass is not timed
            niter.append(d.solver_niter.numpy().mean())
    results[gc] = d.qpos.numpy().copy()
    rerun = np.abs(q_timed - results[gc]).max()
    if ON_GPU:   # contact scenes are not bitwise reproducible on a GPU (Ch 22): float noise only
        assert np.allclose(q_timed, results[gc], atol=1e-4), f"untimed pass drifted: {rerun:.2e}"
    else:        # on CPU the replay is exact
        assert rerun == 0.0, f"untimed pass should repeat the timed one: {rerun:.2e}"
    print(f"graph_conditional={gc!s:5}: {nlaunch:5d} launches recorded, "
          f"mean solver_niter {np.mean(niter):.2f}, {ms:.3f} ms/step, {wsps:,.0f} world-steps/s"
          f"{'' if ON_GPU else ' (CPU dev box)'}, nefc w0 {d.nefc.numpy()[0]}, "
          f"overflow {'none' if not d.overflow.numpy().any() else d.overflow.numpy()}")

if len(results) == 2:
    diff = np.abs(results[True] - results[False]).max()
    print(f"max |qpos(True) - qpos(False)| after {NSTEP} steps: {diff:.2e}")
    print("exact on this run" if diff == 0 else "agrees to float noise (on a GPU, contacts reorder; Ch 22)"
          if diff < 1e-4 else "LARGER than float noise: check the run")

## 6. Toolkit and driver check
Conditional nodes need CUDA >= 12.4 for both the toolkit Warp was built with and the driver. `put_model` only warns; `capture_step` (defined in `my_env.py` above) applies the fallback.

In [ ]:
# Example 24.8 - ex8_toolkit_check.py
"""Example 24.8 - which graph features does this machine have, and what does capture_step do?

Prints the Warp device, the CUDA toolkit Warp was built with and the CUDA driver it found,
wp.is_conditional_graph_supported() (needs both >= 12.4), whether put_model warned
(warp_util.check_toolkit_driver only *warns*; m.opt.graph_conditional stays True), and what
my_env.capture_step decided.
Run: python ex8_toolkit_check.py
"""
import pathlib
import sys
import warnings

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import my_env                           # notebook: written by the %%writefile cell
except ImportError:
    import my_env_step24 as my_env          # examples/24-cuda-graphs/

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""


def version(v):
    return "none" if v is None else f"{v[0]}.{v[1]}"


wp.init()
device = wp.get_device()
print("warp", wp.__version__, "| device:", device, "| is_cuda:", device.is_cuda)
print("CUDA toolkit (Warp build):", version(wp.get_cuda_toolkit_version()))
print("CUDA driver:              ", version(wp.get_cuda_driver_version()))
print("wp.is_conditional_graph_supported():", wp.is_conditional_graph_supported())

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    m = mjw.put_model(mjm)                  # calls warp_util.check_toolkit_driver()
toolkit_warnings = [w for w in caught if "CUDA version < 12.4" in str(w.message)]
print("put_model warned about CUDA < 12.4:", bool(toolkit_warnings))
print("m.opt.graph_conditional after put_model:", m.opt.graph_conditional)

d = mjw.make_data(mjm, nworld=2, nconmax=4, njmax=16)
print("needs_conditional_fallback:", my_env.needs_conditional_fallback(device))
graph = my_env.capture_step(m, d, nsub=1)
print("m.opt.graph_conditional after capture_step:", m.opt.graph_conditional)
for _ in range(3):
    wp.capture_launch(graph)
print(f"3 replays OK: d.time = {d.time.numpy()[0]:.3f}")

if device.is_cuda:
    decision = ("conditional nodes (capture_while / capture_if on the device)"
                if m.opt.graph_conditional else "fallback: fixed iteration counts in the graph")
else:
    decision = "CPU graph (Warp APIC recording; capture_while callbacks replay on the CPU)"
print("decision:", decision)

## 7. Measuring eager vs graph throughput
Warm up, `wp.synchronize()` before reading the clock, report world-steps/s. Uses `my_env.capture_step`. On a GPU the sweep goes to 65536 worlds; set `--plot` or run here to see the log-log chart.

In [ ]:
# Example 24.4 - ex4_eager_vs_graph.py
"""Example 24.4 - world-steps per second vs nworld: eager mjw.step vs a replayed graph.

Uses my_env.capture_step(m, d, nsub) (new in this chapter: it applies the graph-conditional
fallback and records nsub steps into one graph). Warm up first, call wp.synchronize() before
reading the clock, and report world-steps/s = nworld * steps / seconds. The ball starts resting
on the floor, so every timed step has an active contact (collision + constraint solve).
On CPU the numbers are from Warp's CPU device ("on a CPU dev box"), NOT GPU performance;
on a GPU the notebook prints your own numbers.
The log-log plot goes to --plot PNG (default ex4_throughput.png in the current folder; shown
inline in a notebook). On a GPU it also draws the course's CPU numbers (data/ex4_throughput.json)
as dashed reference lines when that file is present.
Run: python ex4_eager_vs_graph.py [--dump-json PATH] [--plot PNG]
"""
import argparse
import json
import pathlib
import sys
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import my_env                           # notebook: written by the %%writefile cell
except ImportError:
    import my_env_step24 as my_env          # examples/24-cuda-graphs/

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 .1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
parser.add_argument("--plot", default=None, help="save the log-log plot to this PNG")
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLDS = [1, 16, 256, 4096, 16384, 65536] if ON_GPU else [1, 4, 16, 64, 256]
NSTEP = 200 if ON_GPU else 30

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)


def timed(fn, n, repeats=3):
    """Seconds per call: best of `repeats` runs of n calls (less noise from other processes)."""
    best = float("inf")
    for _ in range(repeats):
        wp.synchronize()
        t = time.perf_counter()
        for _ in range(n):
            fn()
        wp.synchronize()                    # launches are asynchronous on a GPU
        best = min(best, (time.perf_counter() - t) / n)
    return best


rows = []
print(f"device: {wp.get_device()}  ({'GPU' if ON_GPU else 'CPU dev box, not GPU performance'})")
print(f"{'nworld':>7} {'eager ms':>9} {'graph ms':>9} {'eager w-steps/s':>16} "
      f"{'graph w-steps/s':>16} {'speedup':>8}")
for nw in NWORLDS:
    d = mjw.make_data(mjm, nworld=nw, nconmax=4, njmax=16)
    for _ in range(3):
        mjw.step(m, d)                      # warm-up: compile / load kernels for this size
    eager = timed(lambda: mjw.step(m, d), NSTEP)
    graph = my_env.capture_step(m, d, nsub=1)
    replay = timed(lambda: wp.capture_launch(graph), NSTEP)
    assert not d.overflow.numpy().any(), my_env.describe_overflow(d)
    rows.append({"nworld": nw, "eager_ms": eager * 1e3, "graph_ms": replay * 1e3,
                 "eager_wsps": nw / eager, "graph_wsps": nw / replay})
    print(f"{nw:>7} {eager * 1e3:>9.3f} {replay * 1e3:>9.3f} {nw / eager:>16,.0f} "
          f"{nw / replay:>16,.0f} {eager / replay:>7.1f}x")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"device": str(wp.get_device()), "nstep": NSTEP, "model": "BALL_DROP",
                   "rows": [{k: round(v, 4) if isinstance(v, float) else v for k, v in r.items()}
                            for r in rows]}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

try:
    import matplotlib
    if args.plot or "ipykernel" not in sys.modules:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    nws = [r["nworld"] for r in rows]
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.loglog(nws, [r["eager_wsps"] for r in rows], "o-", label="eager mjw.step")
    ax.loglog(nws, [r["graph_wsps"] for r in rows], "s-", label="graph replay")
    ref = HERE / "data" / "ex4_throughput.json"
    if ON_GPU and ref.exists():             # overlay the course's CPU run for comparison
        cpu = json.loads(ref.read_text())["rows"]
        cnw = [r["nworld"] for r in cpu]
        ax.loglog(cnw, [r["eager_wsps"] for r in cpu], "o--", color="C0", alpha=.5, label="eager, course CPU")
        ax.loglog(cnw, [r["graph_wsps"] for r in cpu], "s--", color="C1", alpha=.5, label="graph, course CPU")
    ax.set_xlabel("nworld")
    ax.set_ylabel("world-steps / s")
    ax.set_title(f"BALL_DROP on {wp.get_device()}")
    ax.legend()
    fig.tight_layout()
    if "ipykernel" in sys.modules and not args.plot:
        plt.show()
    else:
        out = args.plot or "ex4_throughput.png"
        fig.savefig(out, dpi=100)
        print("saved", out)
except ImportError:
    print("skipped: matplotlib not installed")

## 8. Substeps and your own kernels in one graph
`capture_step(m, d, nsub=4, post=...)` records 4 physics steps plus a reward kernel; one replay is one control step.

In [ ]:
# Example 24.7 - ex7_substeps_and_kernels.py
"""Example 24.7 - one graph = 4 physics substeps + your own reward kernel.

An RL control step usually holds several physics steps (the "frame skip") plus kernels you
write (observation, reward, done; Ch 26 and Ch 30). capture_step(m, d, nsub, post) records all of
them into one graph, so a control step is a single wp.capture_launch. We replay it 10 times and
compare with the same work done eagerly: on CPU the difference is exactly 0.
Run: python ex7_substeps_and_kernels.py
"""
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import my_env                           # notebook: written by the %%writefile cell
except ImportError:
    import my_env_step24 as my_env          # examples/24-cuda-graphs/

np.set_printoptions(precision=4, suppress=True)

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSUB = 4                                    # physics steps per control step
NCONTROL = 10                               # control steps (graph replays)


@wp.kernel
def reward_kernel(qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
                  reward: wp.array(dtype=float)):
    worldid = wp.tid()                      # one thread per world
    x = qpos[worldid, 0]
    theta = qpos[worldid, 1]                # 0 = pole upright
    reward[worldid] += 1.0 - theta * theta - 0.01 * x * x - 0.001 * qvel[worldid, 1] ** 2.0


mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
reward = wp.zeros(NWORLD, dtype=float)

qpos0 = np.zeros((NWORLD, mjm.nq), dtype=np.float32)
qpos0[:, 1] = np.linspace(-0.2, 0.2, NWORLD)            # a different pole tilt per world
ctrl0 = np.linspace(0.0, 0.6, NWORLD, dtype=np.float32).reshape(NWORLD, 1)


def restart():
    d.qpos.assign(qpos0)
    d.qvel.zero_()
    d.time.zero_()
    d.ctrl.assign(ctrl0)
    reward.zero_()


def add_reward(m, d):                       # the `post` hook: runs after the NSUB steps
    wp.launch(reward_kernel, dim=d.nworld, inputs=[d.qpos, d.qvel], outputs=[reward])


# eager reference: NCONTROL x (NSUB steps + reward kernel)
restart()
for _ in range(NCONTROL):
    for _ in range(NSUB):
        mjw.step(m, d)
    add_reward(m, d)
wp.synchronize()
eager_r, eager_q = reward.numpy().copy(), d.qpos.numpy().copy()

# graph: the same work recorded once, replayed NCONTROL times
restart()
graph = my_env.capture_step(m, d, nsub=NSUB, post=add_reward)
print(f"after capture: d.time = {d.time.numpy()[0]:.2f}, reward w0 = {reward.numpy()[0]:.1f}"
      "  (nothing ran yet)")
for _ in range(NCONTROL):
    wp.capture_launch(graph)                # 4 physics steps + 1 reward launch per call
wp.synchronize()
graph_r, graph_q = reward.numpy(), d.qpos.numpy()

print(f"{NCONTROL} replays x {NSUB} substeps: d.time = {d.time.numpy()[0]:.2f} s")
print("reward per world (graph):", graph_r)
print("reward per world (eager):", eager_r)
print(f"max |reward diff| = {np.abs(graph_r - eager_r).max():.1e}, "
      f"max |qpos diff| = {np.abs(graph_q - eager_q).max():.1e}")

## Exercise 24.1: a graph replays the same trajectory as eager stepping
Capture one step, replay it NSTEP times, and assert the final `d.qpos` equals NSTEP eager steps from the same start (exact on CPU).

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"/>
  <worldbody>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
</mujoco>
"""
NWORLD, NSTEP = 4, 50
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
if wp.get_device().is_cuda and not wp.is_conditional_graph_supported():
    m.opt.graph_conditional = False                 # Rule G fallback
d = mjw.make_data(mjm, nworld=NWORLD)
q0 = np.zeros((NWORLD, 2), dtype=np.float32)
q0[:, 0] = np.linspace(0.5, 2.0, NWORLD)


def restart():
    d.qpos.assign(q0)
    d.qvel.zero_()
    d.time.zero_()

# 1. restart(), step NSTEP times eagerly, keep a copy of d.qpos in `eager`
# 2. restart(), capture ONE mjw.step, replay it NSTEP times, read d.qpos into `graph`
eager = None  # TODO
graph = None  # TODO
assert isinstance(eager, np.ndarray) and isinstance(graph, np.ndarray), "fill in eager and graph"
assert np.array_equal(graph, eager), np.abs(graph - eager).max()
print("graph replay == eager stepping")

In [ ]:
#@title Exercise 24.1: solution (double-click to show the code)

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"/>
  <worldbody>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
</mujoco>
"""
NWORLD, NSTEP = 4, 50
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
if wp.get_device().is_cuda and not wp.is_conditional_graph_supported():
    m.opt.graph_conditional = False                 # Rule G fallback
d = mjw.make_data(mjm, nworld=NWORLD)
q0 = np.zeros((NWORLD, 2), dtype=np.float32)
q0[:, 0] = np.linspace(0.5, 2.0, NWORLD)


def restart():
    d.qpos.assign(q0)
    d.qvel.zero_()
    d.time.zero_()

restart()
for _ in range(NSTEP):
    mjw.step(m, d)
eager = d.qpos.numpy().copy()

restart()
with wp.ScopedCapture() as capture:       # recording does not advance d
    mjw.step(m, d)
for _ in range(NSTEP):
    wp.capture_launch(capture.graph)
graph = d.qpos.numpy()

assert np.array_equal(graph, eager), np.abs(graph - eager).max()
print("graph replay == eager stepping")   # exact on CPU; on a GPU contact scenes may differ in low bits

## Exercise 24.2: `needs_recapture(field)`
Turn the knob-board rule into code and assert it on five Option fields.

In [ ]:
import mujoco
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

mjm = mujoco.MjModel.from_xml_string("<mujoco><worldbody><body><freejoint/>"
                                     "<geom size='.1'/></body></worldbody></mujoco>")
m = mjw.put_model(mjm)

def needs_recapture(field):
    """True if a graph captured before changing m.opt.<field> must be captured again."""
    ...  # hint: what kind of Python object is getattr(m.opt, field)?

assert needs_recapture("timestep") is False
assert needs_recapture("tolerance") is False
assert needs_recapture("iterations") is True
assert needs_recapture("ls_iterations") is True
assert needs_recapture("integrator") is True
print("needs_recapture OK on 5 fields")

In [ ]:
#@title Exercise 24.2: solution (double-click to show the code)

import mujoco
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

mjm = mujoco.MjModel.from_xml_string("<mujoco><worldbody><body><freejoint/>"
                                     "<geom size='.1'/></body></worldbody></mujoco>")
m = mjw.put_model(mjm)

def needs_recapture(field):
    """True if a graph captured before changing m.opt.<field> must be captured again.

    Array fields are read from device memory at replay (live: change them in place);
    everything else is a Python value read while recording (baked).
    """
    return not isinstance(getattr(m.opt, field), wp.array)

assert needs_recapture("timestep") is False
assert needs_recapture("tolerance") is False
assert needs_recapture("iterations") is True
assert needs_recapture("ls_iterations") is True
assert needs_recapture("integrator") is True
print("needs_recapture OK on 5 fields;", "gravity:", needs_recapture("gravity"),
      "| warn_overflow:", needs_recapture("warn_overflow"))

## Stretch: find the nworld where your GPU saturates

In [ ]:
# Stretch: where does your GPU saturate with graphs?
# Re-run Example 24.4 with a finer sweep, e.g. NWORLDS = [2**k for k in range(0, 18)], and find the
# nworld where graph world-steps/s stops growing (the step time starts to grow with nworld).
# Repeat with box_stack(5) (nconmax=32, njmax=128) instead of BALL_DROP: does it saturate earlier?